# 02 · Signal-quality gate: can we trust this heart-rate reading?

**Role in the two-track design:** this is the measurement-quality gate (`q`) placed in front of the heart-rate risk classifier (notebook 04). It does not predict risk.

**Offline measurement-reliability experiment.** For every four-second optical measurement, predict whether its reported heart rate is within **5 beats/minute** of a simultaneous ECG estimate. Choose the tolerance before examining evaluation results. This label measures HR agreement, not low-HR danger, blackout, drowning or oxygen accuracy. The recommended prototype starts with fixed measurement/quality algorithms; a learned gate needs evidence on the actual chest hardware before use.

Use **WildPPG original participant MATLAB files** (`WildPPG_Part_*.mat`). The study recorded 16 people during ordinary land activities, with synchronized sternum red/infrared PPG, acceleration and lead-I ECG at 128 Hz. Its MAX86141 optics differ from the wearable's MAX30102. The authors' processed convenience benchmark is not a substitute for these original signals. [Dataset](https://siplab.org/projects/WildPPG) · [Original files](https://polybox.ethz.ch/index.php/s/NWTuyNojU7aya1y) · [Source terms](https://github.com/eth-siplab/WildPPG/blob/main/LICENSE.md). The repository lists CC BY-NC-SA 4.0 for data but retains a prepublication restriction; verify applicable terms with the authors before redistribution or commercial reuse. Keep attached study data and exported participant-level records private while resolving those terms.

**Google Colab setup:** open the notebook in Colab with a **CPU** runtime; no GPU, TPU or API key is required. Use **Runtime → Run all** and authorise Google Drive when asked: outputs are saved to `MyDrive/KappaBetaML/outputs/model_b`. The 5.20 GB input cache stays on the runtime disk (`/content/cache`), which Colab wipes when the runtime ends. The reader processes one participant at a time, which fits in a standard runtime's RAM.

- **Automatic download:** leave `ALLOW_DOWNLOAD=True` and run all cells. The downloader retrieves the complete sternum portion of each original MAT file (16 people, 5.20 GB total), validates HTTP ranges/source ETags and resumes interrupted partial downloads within the cache. It does not download the other body sites.
- **Without downloading:** put the original participant MAT files in `MyDrive/KappaBetaML/input/` (or set `LOCAL_DATA` to their folder) and set `ALLOW_DOWNLOAD=False`. The reader finds files recursively and accepts complete originals or the verified sternum-only prefixes created by this downloader. Preserve filenames, original participant IDs and signal structure. Do not substitute a merged `WildPPG.mat` convenience file. Attached files take precedence over downloads; the notebook prints exactly which people were found.
- **Smoke run:** set `QUICK_MODE=True` for up to three five-minute blocks per person. This shortens extraction, **not download size**. To also reduce downloads, set `ONLINE_IDS` to at least six listed IDs. Four is the technical minimum, but a small subset may lack enough class support and is not the benchmark.
- **Main run:** `QUICK_MODE=False`, `ONLINE_IDS=None`, all 16 people, `HOP_SECONDS=4`. Setting `HOP_SECONDS=1` evaluates overlapping firmware-like updates and changes the experiment; save that configuration separately.

All settings can be edited in the next cell. Optional local/automation environment variables are documented there. Outputs go to `MyDrive/KappaBetaML/outputs/model_b` (or `/content/outputs/model_b` with `SAVE_TO_DRIVE = False`), including figures, per-window predictions, coverage audits, models and configuration.

Compare logistic regression, a small random forest, accepting every emitted reading and a simple signal rule. Select the gate and threshold using validation participants, then replay the locked choice on test participants. **This fixed participant split has already been inspected in project development.** Reproducing it is useful, but improvements designed after inspecting it need new untouched participants for an independent effectiveness claim.

**ECG audit (2026-11-16):** the separate v4 counter in notebook 04 overcounted beats. This notebook requires agreement between amplitude and energy detectors. Across 4,560 fixed sampled windows from all 16 people, 2,325 passed its reference checks (51.0%). Among those, median disagreement with an independent ECG periodicity heuristic was 0.42 bpm; 96.0% were within 10 bpm. This does not show the same systematic overcount, and no algorithm or model was changed here. Both references are automatic; this is not an annotated-beat accuracy validation. See `tools/audit_ecg_reference.py` and the results README. Notebook 04 consumes the PPG waveform directly, not this notebook's Maxim HR estimate.


In [ ]:
from pathlib import Path
import json, os, importlib.util, platform
required = {"numpy": "numpy", "pandas": "pandas", "scipy": "scipy",
            "sklearn": "scikit-learn", "joblib": "joblib", "matplotlib": "matplotlib",
            "IPython": "ipython"}
missing = [package for module, package in required.items() if importlib.util.find_spec(module) is None]
if missing:
    raise ImportError("Missing packages: " + ", ".join(missing) +
                      ". Enable Internet and run %pip install " + " ".join(missing) +
                      " in a separate cell, then restart and Run All. Standard Colab runtimes normally include them.")

def env_flag(name, default):
    text = os.environ.get(name)
    if text is None:
        return default
    if text.lower() not in {"0", "1", "true", "false", "yes", "no"}:
        raise ValueError(f"{name} must be true/false or 1/0")
    return text.lower() in {"1", "true", "yes"}
import joblib
import numpy as np
import pandas as pd
from scipy import io, signal
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, brier_score_loss, roc_auc_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from IPython.display import display

# ---- Storage: Google Colab ----
# Outputs go to Google Drive by default (Colab's /content disk is wiped when the runtime ends).
# The 5.2 GB WildPPG download cache stays on the runtime disk; set WILDPPG_CACHE to a Drive path to keep it.
import sys
ON_COLAB = "google.colab" in sys.modules
SAVE_TO_DRIVE = env_flag("KB_SAVE_TO_DRIVE", ON_COLAB)
if ON_COLAB and SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    BASE = Path("/content/drive/MyDrive/KappaBetaML")
else:
    BASE = Path("/content") if ON_COLAB else Path(".")
LOCAL_DATA = Path(os.environ.get("WILDPPG_INPUT", str(BASE / "input")))
OUTPUT = Path(os.environ.get("WILDPPG_OUTPUT", str(BASE / "outputs" / "model_b")))
OUTPUT.mkdir(parents=True, exist_ok=True)
SEED = 20261105
TOLERANCE_BPM = 5.0
TARGET_BAD_ACCEPT_RATE = 0.15  # Validation operating target for this experiment.
# Default uses complete recordings with non-overlapping four-second windows.
# Set HOP_SECONDS=1 to evaluate every firmware update, with overlapping windows.
# Optional quick mode evaluates up to three five-minute blocks in each file.
QUICK_MODE = env_flag("WILDPPG_QUICK", False)
BLOCK_SECONDS, BLOCKS_PER_PERSON = (300 if QUICK_MODE else None), 3
WINDOW_SECONDS, HOP_SECONDS = 4, int(os.environ.get("WILDPPG_HOP_SECONDS", "4"))
if HOP_SECONDS not in {1, 4}:
    raise ValueError("Use a 1-second or 4-second hop and record it as a separate experiment.")
REF_FS, FIFO_FS = 128, 25
# BOXCAR samples 128 Hz photons into 25-Hz non-overlapping time bins.
# Original optical fractions in eight checked files lie on a 2^-19 count grid.
# Project those fractions onto an 18-bit scale. This is a proxy for ADC gain.
ADC_GAIN = float(2**18)
FEATURES = ["optical_hr", "red_ir_corr", "ir_periodicity", "red_periodicity",
            "ir_roughness", "red_roughness", "motion_relative", "jerk_relative",
            "ir_motion_corr", "ir_flat_fraction", "beat_interval_cv", "peak_count"]
ONLINE_IDS = None  # e.g. ["an0", "e61", "fex", "k2s", "kjd", "l38"]
if os.environ.get("WILDPPG_ONLINE_IDS"):
    ONLINE_IDS = [s.strip() for s in os.environ["WILDPPG_ONLINE_IDS"].split(",") if s.strip()]
DOWNLOAD_CACHE = Path(os.environ.get("WILDPPG_CACHE", "/content/cache/wildppg_sternum" if ON_COLAB else str(BASE / "cache" / "wildppg_sternum")))
ALLOW_DOWNLOAD = env_flag("WILDPPG_ALLOW_DOWNLOAD", True)
FIGURES = OUTPUT / "figures"
FIGURES.mkdir(parents=True, exist_ok=True)
print("Research-only HR agreement task; this does not validate SpO2 or blackout detection.")
print({"python": platform.python_version(), "quick_mode": QUICK_MODE,
       "hop_seconds": HOP_SECONDS, "input": str(LOCAL_DATA), "output": str(OUTPUT)})

### Load the original files

Point the input at a directory containing WildPPG participant files. The reader requests the sternum and participant ID only. It verifies each of the six sampling rates and requires synchronized arrays of equal length. A full file may exceed a gigabyte. Process one participant at a time. Quick mode samples the beginning, middle and end of the usable record. Full mode uses the entire recording. Both preserve sample positions. For a cropped participant file, stored times refer to the crop unless an origin is supplied.

In [ ]:
# The original MATLAB v5 file begins with id, notes and the complete sternum matrix.
# Each manifest entry pins the byte immediately after that matrix and the source ETag.
# Downloading this exact prefix preserves every chest sample at its original precision.
from pathlib import Path
import shutil
import time
from urllib.error import URLError
from urllib.request import Request, urlopen
from scipy.io import whosmat

WILDPPG_BASE = "https://polybox.ethz.ch/public.php/dav/files/NWTuyNojU7aya1y/data/"
STERNUM_FILES = {
    "an0": (295671696, "d185bc1ae93e4c9848524e1aab22a39b"),
    "e61": (296257912, "f88e7a783153615a128013363f2d548e"),
    "fex": (320834792, "7273ac5694d4dd2f91aa264e00ddf00a"),
    "k2s": (365403792, "52134984b7904d0145a99c08d15743bd"),
    "kjd": (321207976, "3eefd5de96a73e60bac2e58db0bc1096"),
    "l38": (319555472, "82658918d9bfd74ae511f227a7224343"),
    "n31": (317742648, "226c271e17ecea4ddb7ed844afc4212f"),
    "ngh": (352395448, "e59df9850310836a42df52caa9012a2b"),
    "p5d": (346957672, "a9128090b639d6c0b97532a12bef7c47"),
    "p9p": (299989752, "b35315f9235f273e6b5363aa43e28a30"),
    "qm9": (315823416, "68ef8616be330628e521e38d75d2eaa4"),
    "ssx": (303828216, "fd7dcc41fc08eb4213e2985580705a75"),
    "trh": (345358264, "f82b2148513c5b9e5924146934e30bab"),
    "tz8": (348397048, "82cc5f445aade45c7d3d6aff8d143577"),
    "u7y": (317955896, "de7445d86d476185a62da918eed15c8e"),
    "w4p": (333469688, "b1a570b7c305275c050467702c9ab2db"),
}

def fetch_verified_range(url, start, end, etag):
    headers = {"Range": f"bytes={start}-{end}", "Accept-Encoding": "identity"}
    request = Request(url, headers=headers)
    for attempt in range(3):
        try:
            with urlopen(request, timeout=120) as response:
                served = response.headers.get("Content-Range", "").split("/")[0]
                if (response.status != 206 or served != f"bytes {start}-{end}"
                        or response.headers.get("ETag", "").strip('"') != etag):
                    raise ValueError("ETH file or HTTP range changed. Attach original WildPPG participant MATs.")
                data = response.read(end - start + 2)
                if len(data) != end - start + 1:
                    raise OSError("Incomplete WildPPG transfer")
                return data
        except (URLError, OSError) as exc:
            if attempt == 2:
                raise RuntimeError("Check the runtime's Internet access or upload the original WildPPG MAT files.") from exc
            time.sleep(2 ** attempt)

def ensure_wildppg_inputs(directory, ids=None):
    """Create or resume the chest recording for each requested participant in a cache."""
    root = Path(directory)
    root.mkdir(parents=True, exist_ok=True)
    selected = list(STERNUM_FILES) if ids is None else list(dict.fromkeys(ids))
    if not selected or set(selected) - STERNUM_FILES.keys():
        raise ValueError(f"Select one or more participant IDs from {list(STERNUM_FILES)}")
    paths = []
    for pid in selected:
        expected, etag = STERNUM_FILES[pid]
        name = f"WildPPG_Part_{pid}.mat"
        final = root / name
        part = root / f"{name}.{etag}.part"
        if not final.exists():
            offset = part.stat().st_size if part.exists() else 0
            if offset > expected or shutil.disk_usage(root).free < expected - offset:
                raise OSError(f"Check free space and partial file size for {name}")
            print(f"{pid}: fetching {(expected-offset)/1e6:.0f} MB of sternum data", flush=True)
            with part.open("ab") as output:
                while offset < expected:
                    end = min(offset + 16 * 1024**2, expected) - 1
                    output.write(fetch_verified_range(WILDPPG_BASE + name, offset, end, etag))
                    offset = end + 1
            if [variable for variable, _, _ in whosmat(part, appendmat=False)] != ["id", "notes", "sternum"]:
                raise ValueError(f"Unexpected MATLAB structure in {name}; attach the original recording.")
            part.replace(final)
        if final.stat().st_size != expected:
            raise ValueError(f"Unexpected file in download cache: {final}. Use a separate empty cache directory.")
        paths.append(final)
    return paths

In [ ]:
def original_paths(directory):
    match = sorted(directory.rglob("WildPPG_Part_*.mat")) if directory.exists() else []
    if not match:
        raise FileNotFoundError(f"No WildPPG_Part_*.mat in {directory}. Attach originals or use the download option above.")
    return match

def read_participant(path):
    mat = io.loadmat(path, variable_names=["id", "sternum", "source_offset_seconds"], simplify_cells=True)
    if "id" not in mat or not isinstance(mat.get("sternum"), dict):
        raise ValueError(f"{path.name}: expected original participant fields 'id' and 'sternum'.")
    participant = str(np.asarray(mat["id"]).squeeze()).strip()
    if not participant or participant.lower() in {"nan", "none"}:
        raise ValueError(f"Missing original participant ID in {path.name}")
    stem = mat["sternum"]
    names = ("ppg_r", "ppg_ir", "acc_x", "acc_y", "acc_z", "ecg")
    columns = {}
    for name in names:
        trace = stem.get(name, {})
        if not isinstance(trace, dict) or not {"fs", "v"}.issubset(trace):
            raise ValueError(f"{path.name}: missing sternum.{name}.fs/v")
        if not np.isclose(float(np.asarray(trace["fs"]).squeeze()), REF_FS):
            raise ValueError(f"{path.name}: unsupported {name} sampling rate")
        arr = np.asarray(trace["v"]).squeeze()
        if arr.ndim != 1 or not np.issubdtype(arr.dtype, np.number):
            raise ValueError(f"{path.name}: invalid {name} vector")
        columns[name] = arr
    if len({len(x) for x in columns.values()}) != 1:
        raise ValueError(f"{path.name}: time alignment ambiguous; unequal trace lengths")
    start_seconds = float(mat.get("source_offset_seconds", 0))
    return participant, columns, start_seconds

def choose_blocks(n):
    available = n // REF_FS
    if BLOCK_SECONDS is None or available <= BLOCK_SECONDS * BLOCKS_PER_PERSON:
        return [(0, available)]
    starts = np.linspace(0, available - BLOCK_SECONDS, BLOCKS_PER_PERSON).astype(int)
    return [(i, i + BLOCK_SECONDS) for i in starts]

def as_fifo(raw):
    """Average samples in completed native time bins; no future bin contributes."""
    n = len(raw) * FIFO_FS // REF_FS
    edges = np.arange(n + 1, dtype=np.int64) * REF_FS // FIFO_FS
    values = np.asarray(raw, dtype=np.float64)
    cumulative = np.r_[0., np.cumsum(np.where(np.isfinite(values), values, 0.))]
    invalid = np.r_[0, np.cumsum(~np.isfinite(values))]
    out = np.diff(cumulative[edges]) / np.diff(edges)
    out[np.diff(invalid[edges]) != 0] = np.nan
    return out

### Replay the HR calculation

The ESP32's Maxim routine uses the infrared channel to compute HR from a trailing 100-sample window at 25 Hz, with a new estimate every second. The implementation below reproduces its integer mean, partial four-point smoothing, clamped detection threshold, ordering of peaks and integer BPM calculation. At a plateau reaching the very end, the C++ source can read past its signal buffer. This replay abstains in that case. The red channel helps assess signal agreement.

This is a **replay on a different optical sensor**. WildPPG uses a MAX86141, whereas the wearable uses a MAX30102. The firmware now reads raw 100 Hz samples with on-chip averaging off and records each one. Before the Maxim routine it averages every four raw samples in software (`PpgDecimator.h`), so the routine's 25 Hz input is unchanged. Boxing the 128-Hz source approximates that 25 Hz cadence. Beat-level features from the raw 100 Hz stream are a separate future experiment (see notebook 04's HRV comparison). The replay retains the study sensor's amplitude units scaled by `ADC_GAIN`; raw thresholds are gain sensitive. Wearable-specific contact counts, clipping and FIFO resets need tests on the hardware. This notebook keeps reading eligibility and reference eligibility visible throughout. [SparkFun/Maxim reference implementation v1.1.2](https://github.com/sparkfun/SparkFun_MAX3010x_Sensor_Library/blob/v1.1.2/src/spo2_algorithm.cpp)

In [ ]:
def replay_ir_hr(ir):
    """Return (BPM, peak positions, reason) for one 100-sample proxy window."""
    if len(ir) != 100 or not np.isfinite(ir).all():
        return np.nan, [], "optical_missing"
    # Integer DC removal means a constant offset has no effect. Preserve AC gain.
    counts = np.rint(ir * ADC_GAIN).astype(np.int64)
    counts -= counts.min()  # Integer DC shift preserves the Maxim HR branch.
    if np.max(counts) > 2**24 or ADC_GAIN <= 0:
        return np.nan, [], "optical_scale"
    x = counts.sum() // 100 - counts
    smoothed = np.convolve(x, np.ones(4, dtype=int), mode="valid")[:96]
    x[:96] = np.trunc(smoothed / 4.).astype(np.int64)
    threshold = int(np.clip(np.trunc(x.sum() / 100.), 30, 60))
    candidates = []
    i = 1
    while i < 99:
        if x[i] > threshold and x[i] > x[i - 1]:
            end = i + 1
            while end < 100 and x[end] == x[i]:
                end += 1
            if end == 100:
                return np.nan, [], "trailing_plateau"
            if x[i] > x[end] and len(candidates) < 15:
                candidates.append(i)
                i = end + 1
            else:
                i = end
        else:
            i += 1
    peaks = []
    for pos in sorted(candidates, key=lambda p: -x[p]):
        if pos > 3 and all(abs(pos - retained) > 4 for retained in peaks):
            peaks.append(pos)
    peaks.sort()
    if len(peaks) < 2:
        return np.nan, peaks, "no_optical_estimate"
    avg_interval = (peaks[-1] - peaks[0]) // (len(peaks) - 1)
    return 1500 // avg_interval, peaks, "emitted"

def corr(a, b):
    a, b = a - a.mean(), b - b.mean()
    denom = np.linalg.norm(a) * np.linalg.norm(b)
    return float(np.dot(a, b) / denom) if denom > 0 else 0.

def pulse_features(x, lag):
    x = signal.detrend(x, type="linear")
    periodicity = corr(x[lag:], x[:-lag]) if 2 <= lag <= len(x) // 2 else 0.
    roughness = np.std(np.diff(x)) / max(np.std(x), 1e-9)
    return periodicity, roughness

def quality_features(red, ir, acc, hr, peaks):
    lag = int(round(60 * FIFO_FS / hr))
    ir_p, ir_r = pulse_features(ir, lag)
    red_p, red_r = pulse_features(red, lag)
    acc_mag = np.linalg.norm(acc, axis=1)
    accel_scale = max(np.median(acc_mag), 1e-9)
    moving = acc - np.mean(acc, axis=0)
    ir_ac, red_ac = signal.detrend(ir), signal.detrend(red)
    rr = np.diff(peaks)
    return dict(optical_hr=hr, red_ir_corr=corr(red_ac, ir_ac),
                ir_periodicity=ir_p, red_periodicity=red_p,
                ir_roughness=ir_r, red_roughness=red_r,
                motion_relative=np.sqrt(np.mean(moving ** 2)) / accel_scale,
                jerk_relative=np.sqrt(np.mean(np.diff(acc, axis=0) ** 2)) / accel_scale,
                ir_motion_corr=abs(corr(ir_ac, acc_mag)),
                ir_flat_fraction=np.mean(np.diff(ir) == 0),
                beat_interval_cv=np.std(rr) / max(np.mean(rr), 1.), peak_count=len(peaks))

### Build the reference from ECG only

Reference handling is independent of the optical prediction. Two detectors locate candidate QRS complexes using different signal features. The first searches filtered ECG amplitude in its dominant polarity. The second searches integrated derivative energy and refines its timing locally. A window is reference-eligible when both find at least three matching peaks within 80 ms and the complete intervals fall inside a broad 0.25–2.5 s engineering envelope. Mean complete RR intervals from matched beats give the reference BPM. The offline ECG processing can use both sides of each QRS peak because it only creates labels. A 0.5 s edge exclusion avoids boundary transients.

These checks discard some irregular, very slow and motion-contaminated rhythms. They are an automated reference heuristic. Inspect the example overlays and eligibility counts. Independently adjudicated beats are preferable for final validation. The model features use only optical and motion observations available by the end of the window.

In [ ]:
def ecg_peaks(ecg):
    """Offline reference only. Process contiguous finite runs, preserving gaps."""
    raw = np.asarray(ecg, dtype=float)
    good = np.isfinite(raw)
    edges = np.flatnonzero(np.diff(np.r_[False, good, False]))
    first, second, valid = [], [], np.zeros(len(raw), dtype=bool)
    sos = signal.butter(3, [5., 20.], fs=REF_FS, btype="bandpass", output="sos")
    for start, stop in zip(edges[::2], edges[1::2]):
        if stop - start < 5 * REF_FS:
            continue
        filtered = signal.sosfiltfilt(sos, raw[start:stop])
        if np.ptp(filtered) < 1e-10:
            continue
        # Assess amplitude in short overlapping regions so hours of moving data
        # cannot set a common amplitude threshold for rest and exertion.
        for begin in range(0, len(filtered), 30 * REF_FS):
            end = min(begin + 30 * REF_FS, len(filtered))
            lo, hi = max(0, begin - REF_FS), min(len(filtered), end + REF_FS)
            q = filtered[lo:hi]
            if len(q) < REF_FS:
                continue
            sign = 1 if np.percentile(q, 99) >= -np.percentile(q, 1) else -1
            a = sign * q
            prominence = max(1e-12, 0.22 * np.percentile(np.abs(q), 99))
            peaks_a = signal.find_peaks(a, distance=int(.25 * REF_FS), prominence=prominence)[0]
            weight = np.diff(q, prepend=q[0]) ** 2
            kernel = np.ones(9) / 9
            energy = np.convolve(weight, kernel, mode="same")
            level = max(1e-15, 0.15 * np.percentile(energy, 98))
            peaks_b = signal.find_peaks(energy, height=level, prominence=level * .5,
                                        distance=int(.25 * REF_FS))[0]
            radius = int(.10 * REF_FS)
            refined = []
            for p in peaks_b:
                l, h = max(0, p - radius), min(len(q), p + radius + 1)
                refined.append(l + int(np.argmax(np.abs(q[l:h]))))
            for source, dest in [(peaks_a, first), (refined, second)]:
                dest.extend(start + lo + int(p) for p in source if begin <= lo + p < end)
        valid[start + REF_FS // 2:stop - REF_FS // 2] = True
    return np.sort(np.unique(first)).astype(int), np.sort(np.unique(second)).astype(int), valid

def reference_hr(first, second, valid, l, h):
    if not valid[l:h].all():
        return np.nan, "ecg_gap_or_edge"
    a = first[np.searchsorted(first, l):np.searchsorted(first, h)]
    b = second[np.searchsorted(second, l):np.searchsorted(second, h)]
    if min(len(a), len(b)) < 3:
        return np.nan, "ecg_few_beats"
    if len(a) != len(b) or np.max(np.abs(a - b)) > round(.08 * REF_FS):
        return np.nan, "ecg_disagreement"
    rr = np.diff(a) / REF_FS
    if np.any((rr < .25) | (rr > 2.5)):
        return np.nan, "ecg_interval_outside_range"
    return 60 / np.mean(rr), "qualified"

In [ ]:
def extract_rows(participant, data, offset):
    rows, examples = [], []
    for lo, hi in choose_blocks(len(data["ecg"])):
        if hi - lo < WINDOW_SECONDS + 2:
            continue
        indices = slice(lo * REF_FS, hi * REF_FS)
        ir, red = (as_fifo(data[k][indices]) for k in ["ppg_ir", "ppg_r"])
        acc = np.column_stack([as_fifo(data[k][indices]) for k in ["acc_x", "acc_y", "acc_z"]])
        ecg = data["ecg"][indices]
        first, second, ecg_valid = ecg_peaks(ecg)
        for stop in range(WINDOW_SECONDS, hi - lo + 1, HOP_SECONDS):
            begin = stop - WINDOW_SECONDS
            a, b = begin * REF_FS, stop * REF_FS
            idx = slice(begin * FIFO_FS, stop * FIFO_FS)
            ref, ref_status = reference_hr(first, second, ecg_valid, a, b)
            rw, iw, mw = red[idx], ir[idx], acc[idx]
            hr, peaks, status = replay_ir_hr(iw)
            if not np.isfinite(rw).all():
                hr, status = np.nan, "red_missing"
            row = dict(participant=participant, end_s=offset + lo + stop,
                       reference_bpm=ref, reference_status=ref_status,
                       optical_hr=hr, optical_status=status, features_available=False)
            if np.isfinite(hr) and np.isfinite(mw).all():
                row.update(quality_features(rw, iw, mw, hr, peaks))
                row["features_available"] = all(np.isfinite(row[k]) for k in FEATURES)
            rows.append(row)
            if np.isfinite(ref) and np.isfinite(hr):
                kind = "agreement" if abs(ref - hr) <= TOLERANCE_BPM else "disagreement"
                if kind not in {e["kind"] for e in examples}:
                    examples.append(dict(participant=participant, kind=kind, ecg=ecg[a:b].copy(),
                                         primary=first[(first >= a) & (first < b)] - a,
                                         secondary=second[(second >= a) & (second < b)] - a,
                                         ir=iw.copy(), peaks=peaks, reference=ref, optical=hr))
    return rows, examples

attached = sorted(LOCAL_DATA.rglob("WildPPG_Part_*.mat")) if LOCAL_DATA.exists() else []
if attached:
    paths = original_paths(LOCAL_DATA)
elif ALLOW_DOWNLOAD:
    paths = ensure_wildppg_inputs(DOWNLOAD_CACHE, ONLINE_IDS)
else:
    raise FileNotFoundError("Attach WildPPG MAT files or enable the online download.")
print(f"Using {len(paths)} participant files; attached files take precedence over ONLINE_IDS.", flush=True)
if len(paths) != 16:
    print("Subset run: use all 16 original participants for the reference benchmark.")
source_manifest = pd.DataFrame([{
    "filename": p.name, "bytes": p.stat().st_size,
    "source_url": WILDPPG_BASE + p.name,
    "acquisition": "attached; structure/rate checked" if attached else "HTTP range and source ETag checked",
    "pinned_source_etag": STERNUM_FILES.get(p.stem.rsplit("_", 1)[-1], (None, None))[1]
} for p in paths])
source_manifest.to_csv(OUTPUT / "source_manifest.csv", index=False)
all_rows, audit_examples, seen = [], {}, set()
for path in paths:
    pid, recording, offset = read_participant(path)
    if pid in seen:
        raise ValueError(f"Duplicate participant {pid!r}. Use one original file per participant.")
    seen.add(pid)
    rows, audit_examples[pid] = extract_rows(pid, recording, offset)
    all_rows.extend(rows)
    print(f"{pid}: {len(rows):,} candidate windows from {path.name}", flush=True)
    del recording, rows
if not all_rows:
    raise ValueError("No four-second windows. Check recording duration and fs.")
windows = pd.DataFrame.from_records(all_rows).reindex(
    columns=list(dict.fromkeys(["participant", "end_s", "reference_bpm", "reference_status",
                                "optical_status", "features_available"] + FEATURES)))
windows["abs_error"] = (windows.optical_hr - windows.reference_bpm).abs()
windows["eligible"] = windows.features_available & windows.reference_bpm.notna()
windows["agree"] = np.where(windows.eligible, windows.abs_error <= TOLERANCE_BPM, np.nan)
del all_rows
display(pd.crosstab(windows.optical_status, windows.reference_status, margins=True))
display(windows.groupby("participant").agg(
    attempted=("end_s", "size"), optical_emitted=("optical_hr", "count"),
    ecg_qualified=("reference_bpm", "count"), eligible=("eligible", "sum"),
    agreement_on_eligible=("agree", "mean")))
# Saves per-person summary only. Preserve subject privacy and source terms if exporting more.
windows.groupby(["participant", "optical_status", "reference_status"], observed=True).size().rename(
    "n_windows").to_csv(OUTPUT / "window_eligibility.csv")

### Hold people out before fitting or tuning

The model sees only qualified windows with an available optical estimate and all runtime features. This defines the denominator for classification metrics. Every attempted window remains in the operational coverage calculation, including unavailable readings. Participants are assigned once using a fixed seed independent of outcomes. The holdout may contain only one participant in a small run. Use all 16 for the main experiment and review results by individual. Check reference quality on training examples before interpreting scores.

In [ ]:
ids = np.array(sorted(windows.participant.unique()))
if len(ids) < 4:
    raise ValueError(f"Feature extraction succeeded for {len(ids)} participants. Attach at least four distinct people for training, validation and testing.")
order = np.random.default_rng(SEED).permutation(ids)
n_hold = max(1, round(0.20 * len(order)))
test_people = set(order[:n_hold])
val_people = set(order[n_hold:2 * n_hold])
train_people = set(order[2 * n_hold:])
assert len(train_people) >= 2 and not (train_people & val_people or train_people & test_people or val_people & test_people)
split_names = {p: name for name, group in [("train", train_people), ("validation", val_people),
                                           ("test", test_people)] for p in group}
windows["split"] = windows.participant.map(split_names)
qualified = windows.loc[windows.eligible].copy()
qualified["agree"] = qualified.agree.astype(int)
train, val, test = (qualified[qualified.split.eq(s)] for s in ["train", "validation", "test"])
print({s: sorted(g) for s, g in [("train", train_people), ("validation", val_people), ("test", test_people)]})
if train.agree.nunique() < 2 or val.agree.nunique() < 2 or test.empty:
    raise ValueError("Insufficient class support or eligible held-out windows. Review printed quality counts and extend the data or sampling blocks. Keep the participant split fixed.")
import matplotlib.pyplot as plt
available_examples = [ex for person in sorted(train_people) for ex in audit_examples[person]]
for kind in ["agreement", "disagreement"]:
    ex = next((item for item in available_examples if item["kind"] == kind), None)
    if ex is None:
        continue
    fig, ax = plt.subplots(2, 1, figsize=(8, 3.2), sharex=True, constrained_layout=True)
    ax[0].plot(np.arange(len(ex["ecg"])) / REF_FS, ex["ecg"], color="0.35")
    for tag, marker in [("primary", "o"), ("secondary", "x")]:
        points = ex[tag]
        ax[0].scatter(points / REF_FS, ex["ecg"][points], marker=marker, s=35,
                      facecolors="none" if tag == "primary" else None, label=tag)
    ax[0].legend(); ax[0].set_ylabel("ECG"); ax[0].set_title(f"{ex['participant']} | {kind} | ECG {ex['reference']:.1f} bpm")
    ax[1].plot(np.arange(len(ex["ir"])) / FIFO_FS, ex["ir"], color="teal")
    pts = np.asarray(ex["peaks"], dtype=int)
    ax[1].scatter(pts / FIFO_FS, ex["ir"][pts], s=15, color="black")
    ax[1].set(xlabel="Seconds in window", ylabel="Infrared", title=f"Optical estimate {ex['optical']:.0f} bpm")
    fig.savefig(FIGURES / f"reference_example_{kind}.png", dpi=150, bbox_inches="tight")
    plt.show()
    plt.close(fig)

### Compare gates on the validation participants

An **inaccurate reading** exceeds the predeclared absolute BPM tolerance. The false-accept rate is the fraction of all inaccurate, evaluable readings that the gate accepts. Error among accepted readings answers a different question and has a different denominator. For each gate, select the threshold with greatest coverage subject to the validation false-accept limit. If none of the observed thresholds qualifies, the policy abstains on every score. This is an exploratory trade-off, not a deployment specification. Do not use test subjects to set it.

Fit logistic regression and a random forest with fixed hyperparameters. Each participant contributes the same total training weight. Their fitting losses optimize measurement agreement. The simple rule ranks readings by the weaker of two measurements: red/IR shape agreement and IR periodicity at the reported pulse interval. This rule needs no training.

In [ ]:
# The reference, split, time, person ID and absolute error never enter the predictor.
X = train[FEATURES]
y = train.agree
counts = train.participant.value_counts()
weights = train.participant.map(lambda p: len(train) / (len(counts) * counts[p])).to_numpy()
logistic = make_pipeline(StandardScaler(), LogisticRegression(C=1, max_iter=1000, random_state=SEED))
logistic.fit(X, y, standardscaler__sample_weight=weights, logisticregression__sample_weight=weights)
forest = RandomForestClassifier(n_estimators=150, max_depth=8, min_samples_leaf=15,
                                random_state=SEED, n_jobs=-1)
forest.fit(X, y, sample_weight=weights)
models = {"logistic": logistic, "random_forest": forest}

def scores_for(frame):
    rule = np.minimum(np.abs(frame.red_ir_corr), np.maximum(0., frame.ir_periodicity))
    return {"accept_all": np.ones(len(frame)), "signal_rule": rule.to_numpy(),
            **{name: model.predict_proba(frame[FEATURES])[:, 1] for name, model in models.items()}}

def select_threshold(score, label, limit):
    table = pd.DataFrame({"s": score, "good": label, "bad": 1 - label})
    totals = table.groupby("s")[["good", "bad"]].sum().sort_index(ascending=False).cumsum()
    eligible = totals[totals.bad / table.bad.sum() <= limit]
    if eligible.empty:
        return 1.000001
    return float(eligible.index[-1])

def ratio(a, b):
    return a / b if b else np.nan

def gate_report(frame, score, threshold, total_attempts, model_name):
    label = frame.agree.to_numpy(dtype=bool)
    accepted = np.asarray(score) >= threshold
    good_accepted = np.sum(accepted & label)
    bad_accepted = np.sum(accepted & ~label)
    estimate_error = frame.abs_error.to_numpy()
    too_low = frame.optical_hr.to_numpy() < frame.reference_bpm.to_numpy() - TOLERANCE_BPM
    return dict(model=model_name, participants=frame.participant.nunique(), attempts=total_attempts,
                evaluable=len(label), good=int(label.sum()), bad=int((~label).sum()),
                accepted=int(accepted.sum()), false_accepts=int(bad_accepted), threshold=threshold,
                false_accept_rate=ratio(bad_accepted, (~label).sum()),
                good_retained=ratio(good_accepted, label.sum()),
                error_among_accepted=ratio(bad_accepted, accepted.sum()),
                coverage_evaluable=ratio(accepted.sum(), len(label)),
                verified_accepted_per_attempt=ratio(accepted.sum(), total_attempts),
                falsely_low_accepted=int(np.sum(accepted & too_low)),
                falsely_low_total=int(too_low.sum()),
                false_low_accept_rate=ratio(np.sum(accepted & too_low), too_low.sum()),
                accepted_mae_bpm=float(np.mean(estimate_error[accepted])) if np.any(accepted) else np.nan,
                auc=roc_auc_score(label, score) if len(label) and label.min() != label.max() else np.nan,
                pr_auc_agreement=average_precision_score(label, score) if len(label) and label.min() != label.max() else np.nan,
                brier_agreement=brier_score_loss(label, score) if model_name in models and len(label) else np.nan)

val_scores = scores_for(val)
thresholds = {name: 0.0 if name == "accept_all" else select_threshold(score, val.agree.to_numpy(),
                                                                     TARGET_BAD_ACCEPT_RATE)
              for name, score in val_scores.items()}
val_attempts = int(windows.split.eq("validation").sum())
validation = pd.DataFrame([gate_report(val, scores, thresholds[name], val_attempts, name)
                           for name, scores in val_scores.items()])
display(validation.round(3))
# Predeclare tie order by complexity, then prefer most evaluable coverage.
candidates = validation[validation.model.isin(["signal_rule", "logistic", "random_forest"])].copy()
candidates["simplicity"] = candidates.model.map({"signal_rule": 0, "logistic": 1, "random_forest": 2})
selected = candidates.sort_values(["coverage_evaluable", "simplicity"], ascending=[False, True]).iloc[0].model
print("Validation-selected gate:", selected, "minimum score:", thresholds[selected])
print("Features for all fitted models:", FEATURES)

### Evaluate the locked gate and save the result

Read **false accepts out of bad readings**, **good readings retained** and **coverage** together. A gate can appear accurate by accepting very few readings. The denominator for `verified_accepted_per_attempt` is all attempted windows, including ungradable ECG. It measures how much data received both an acceptance and an independent check. The operational acceptance column below also evaluates readings whose ECG was ungradable, using only the runtime features.

A higher score ranks agreement with this ECG-based HR target in these data; it is not automatically a calibrated probability. Validate the physical chest MAX30102 against synchronized independent ECG or an aquatic reference on new people before selecting thresholds for the wearable. A separate oxygen-reference experiment is needed for SpO₂ confidence.

The reference-HR table reports test coverage and inaccurate readings accepted in each pulse range for the selected gate. A slow pulse produces fewer beats in four seconds. A window with fewer than three matched ECG beats has no reference HR under this protocol and appears in the unavailable group. These ranges are descriptive bins. They do not set the wearable's warning thresholds.

In [ ]:
test_scores = scores_for(test)
test_attempts = int(windows.split.eq("test").sum())
comparison = pd.DataFrame([gate_report(test, scores, thresholds[name], test_attempts, name)
                           for name, scores in test_scores.items()])
# Includes every runtime-qualified output, even when ECG is missing or unusable.
operational = windows.loc[windows.split.eq("test") & windows.features_available]
operational_scores = scores_for(operational)
comparison["operational_accept_per_attempt"] = [np.sum(operational_scores[n] >= thresholds[n]) / test_attempts
                                                for n in comparison.model]
display(comparison.round(3))
by_person = []
for pid in sorted(test_people):
    individual = test[test.participant.eq(pid)]
    idx = test.participant.eq(pid).to_numpy()
    n_trials = int(windows.participant.eq(pid).sum())
    entry = gate_report(individual, test_scores[selected][idx], thresholds[selected], n_trials, selected)
    by_person.append(dict(participant=pid, **entry))
display(pd.DataFrame(by_person).round(3))

# Describe observable low HR separately from windows that ECG could not grade.
held_out = windows.loc[windows.split.eq("test")].copy()
held_out["accepted"] = False
held_out.loc[operational.index, "accepted"] = operational_scores[selected] >= thresholds[selected]
bands = pd.cut(held_out.reference_bpm, [-np.inf, 50, 60, np.inf],
              labels=["below 50", "50 to below 60", "60 or above"], right=False)
held_out["reference_hr_band"] = bands.cat.add_categories("reference unavailable").fillna("reference unavailable")
by_hr = []
for band, group in held_out.groupby("reference_hr_band", observed=False):
    evaluable = group[group.eligible]
    bad = evaluable.agree.eq(0)
    good = evaluable.agree.eq(1)
    false_accepts = int((bad & evaluable.accepted).sum())
    by_hr.append(dict(reference_hr_band=band, attempts=len(group), participants=group.participant.nunique(),
                      evaluable=len(evaluable), inaccurate=int(bad.sum()), false_accepts=false_accepts,
                      inaccurate_accept_rate=ratio(false_accepts, int(bad.sum())),
                      good_retained=ratio(int((good & evaluable.accepted).sum()), int(good.sum())),
                      accepted_all=int(group.accepted.sum()),
                      operational_coverage=ratio(int(group.accepted.sum()), len(group))))
hr_report = pd.DataFrame(by_hr)
display(hr_report.round(3))
hr_report.to_csv(OUTPUT / "test_reference_hr_bands.csv", index=False)

validation.to_csv(OUTPUT / "validation_gates.csv", index=False)
comparison.to_csv(OUTPUT / "test_gates.csv", index=False)
pd.DataFrame(by_person).to_csv(OUTPUT / "test_participants.csv", index=False)
bundle = {"models": models, "features": FEATURES, "selected_gate": selected, "thresholds": thresholds,
          "rule": "min(abs(red_ir_corr), max(0, ir_periodicity))"}
joblib.dump(bundle, OUTPUT / "model_b_research.joblib")
config = dict(task="optical HR agreement with ECG", tolerance_bpm=TOLERANCE_BPM,
              selected=selected, thresholds=thresholds, validation_bad_accept_target=TARGET_BAD_ACCEPT_RATE,
              window_seconds=WINDOW_SECONDS, hop_seconds=HOP_SECONDS, proxy_gain=ADC_GAIN,
              block_seconds=BLOCK_SECONDS, blocks_per_person=BLOCKS_PER_PERSON, seed=SEED,
              train=sorted(train_people), validation=sorted(val_people), test=sorted(test_people),
              features=FEATURES, versions={})
import sklearn, scipy
config["versions"] = {"scikit-learn": sklearn.__version__, "scipy": scipy.__version__, "numpy": np.__version__}
(OUTPUT / "model_b_configuration.json").write_text(json.dumps(config, indent=2))
print(f"Research models, thresholds and full evaluation tables: {OUTPUT}")

# Exports are research artifacts, never an automatic authorization to use the gate.
config.update(status="offline research; not validated for wearable deployment",
              calibration_status="uncalibrated agreement scores on a participant-weighted training mix",
              inspected_split=True, quick_mode=QUICK_MODE,
              reference_limitations="automated ECG agreement filtering excludes many slow/irregular/noisy windows",
              device_transfer="WildPPG MAX86141 sternum to MAX30102 chest requires new reference validation",
              source_manifest="source_manifest.csv", figures="figures")
(OUTPUT / "model_b_configuration.json").write_text(json.dumps(config, indent=2))
pd.DataFrame([{"participant": p, "split": s} for p, s in sorted(split_names.items())]).to_csv(
    OUTPUT / "participant_splits.csv", index=False)
loaded_bundle = joblib.load(OUTPUT / "model_b_research.joblib")  # Only load artifacts you generated/trust.
for name in models:
    np.testing.assert_allclose(loaded_bundle["models"][name].predict_proba(test[FEATURES].iloc[:3]),
                               models[name].predict_proba(test[FEATURES].iloc[:3]))
print("Saved model reload checks passed. A validation-selected gate is not a validated safety gate.")


### Inspect useful performance, failures and coverage

These figures are diagnostic, not a second opportunity to select the test threshold. The gate trade-off curves sweep thresholds **for description**; their marked operating points use the locked validation thresholds. The selected gate must be judged using **bad readings accepted / all bad readings**, **bad readings / accepted readings**, good readings retained, error magnitude and monitoring coverage together.

Calibration is descriptive agreement frequency on this test population, not a blackout or SpO₂ probability. Overlapping/nearby windows are correlated, so no window-independent confidence intervals are shown. The CSV includes calibration-bin support; scarce subgroups should not be interpreted from smooth curves. A constant training-prevalence score is included for the probability-loss benchmark. Do not fit a calibrator to these test records.

Time replays use the first ten minutes of every test participant, ordered by ID, without selecting for success. Gaps are broken rather than joined. HR-band plots include the reference-unavailable category so the lack of validated low-HR performance stays visible. Every recorded replay window remains in coverage denominators, including ungradable ECG. Exported participant-level data retain source-license obligations.


In [ ]:
from sklearn.metrics import precision_recall_curve, log_loss

def save_figure(fig, name):
    fig.savefig(FIGURES / f"{name}.png", dpi=150, bbox_inches="tight")
    plt.show()
    plt.close(fig)

# All attempts, not just the windows that an offline ECG heuristic can grade.
support = windows.groupby(["split", "participant"], sort=True).agg(
    attempts=("end_s", "size"), runtime_features=("features_available", "sum"),
    reference_available=("reference_bpm", "count"), evaluable=("eligible", "sum"),
    good=("agree", "sum"))
support.to_csv(OUTPUT / "support_by_participant.csv")
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5), constrained_layout=True)
labels = [f"{p} ({s})" for s, p in support.index]
x = np.arange(len(support))
attempts = support.attempts.to_numpy()
eligible_fraction = support.evaluable.to_numpy() / attempts
runtime_only = (support.runtime_features - support.evaluable).to_numpy() / attempts
axes[0].bar(x, eligible_fraction, label="Runtime + ECG reference available", color="teal")
axes[0].bar(x, runtime_only, bottom=eligible_fraction, label="Runtime available; ECG ungradable", color="goldenrod")
axes[0].bar(x, 1 - eligible_fraction - runtime_only, bottom=eligible_fraction + runtime_only,
            label="Runtime features unavailable", color="lightgray")
axes[0].set(xticks=x, xticklabels=labels, ylabel="Fraction of attempted windows", ylim=(0, 1),
            title="Monitoring and reference coverage")
axes[0].tick_params(axis="x", rotation=70)
axes[0].legend(fontsize=8)
axes[1].bar(x, attempts, color="slategray", label="All attempts")
axes[1].bar(x, support.good.to_numpy(), color="teal", label=f"Agreement within {TOLERANCE_BPM:g} bpm")
axes[1].set(xticks=x, xticklabels=labels, ylabel="Window count", title="Label support (correlated windows)")
axes[1].tick_params(axis="x", rotation=70)
axes[1].legend(fontsize=8)
save_figure(fig, "01_support_and_coverage")

def gate_curve(frame, score, model_name, partition):
    score = np.asarray(score, dtype=float)
    label = frame.agree.to_numpy(dtype=bool)
    cutoffs = np.r_[np.inf, np.unique(np.quantile(score, np.linspace(0, 1, 201)))[::-1]]
    rows = []
    for cutoff in cutoffs:
        keep = score >= cutoff
        bad_n, good_n = int(np.sum(keep & ~label)), int(np.sum(keep & label))
        rows.append(dict(split=partition, model=model_name, threshold=cutoff,
                         false_accept_rate=ratio(bad_n, int((~label).sum())),
                         good_retained=ratio(good_n, int(label.sum())),
                         coverage_evaluable=ratio(int(keep.sum()), len(label)),
                         error_among_accepted=ratio(bad_n, int(keep.sum()))))
    return pd.DataFrame(rows)

curves = pd.concat([gate_curve(frame, scores[name], name, partition)
                    for partition, frame, scores in [("validation", val, val_scores), ("test", test, test_scores)]
                    for name in scores], ignore_index=True)
curves.to_csv(OUTPUT / "gate_operating_curves.csv", index=False)
fig, axes = plt.subplots(2, 2, figsize=(11, 8), constrained_layout=True)
colors = dict(zip(test_scores, ["gray", "darkorange", "royalblue", "teal"]))
for col, (partition, locked) in enumerate([("validation", validation), ("test", comparison)]):
    for name in test_scores:
        curve = curves[curves.split.eq(partition) & curves.model.eq(name)]
        point = locked.loc[locked.model.eq(name)].iloc[0]
        axes[0, col].plot(curve.false_accept_rate, curve.good_retained, color=colors[name], label=name,
                         linestyle="None" if name == "accept_all" else "-",
                         marker="o" if name == "accept_all" else None)
        axes[0, col].scatter(point.false_accept_rate, point.good_retained, color=colors[name], marker="x", s=55)
        axes[1, col].plot(curve.coverage_evaluable, curve.error_among_accepted, color=colors[name], label=name,
                         linestyle="None" if name == "accept_all" else "-",
                         marker="o" if name == "accept_all" else None)
        axes[1, col].scatter(point.coverage_evaluable, point.error_among_accepted, color=colors[name], marker="x", s=55)
    axes[0, col].axvline(TARGET_BAD_ACCEPT_RATE, ls="--", color="black", lw=.8, label="Validation target")
    axes[0, col].set(title=f"{partition}: retention vs bad acceptance", xlabel="Bad accepted / all bad",
                     ylabel="Good retained / all good", xlim=(0, 1), ylim=(0, 1))
    axes[1, col].set(title=f"{partition}: accepted-error burden", xlabel="Accepted / evaluable",
                     ylabel="Bad / accepted", xlim=(0, 1), ylim=(0, 1))
    for row in [0, 1]:
        axes[row, col].grid(alpha=.2)
axes[0, 0].legend(fontsize=8)
save_figure(fig, "02_gate_tradeoffs_locked_thresholds")

constant_probability = float(np.average(train.agree.to_numpy(), weights=weights))
probability_rows, calibration_rows = [], []
fig, axes = plt.subplots(1, 3, figsize=(14, 4), constrained_layout=True)
if test.agree.nunique() == 2:
    for name, score in test_scores.items():
        precision, recall, _ = precision_recall_curve(test.agree, score)
        axes[0].step(recall, precision, where="post", color=colors[name], label=name)
    axes[0].axhline(float(test.agree.mean()), color="black", ls="--", label="Test prevalence")
else:
    axes[0].text(.5, .5, "Only one class: PR comparison unavailable", ha="center", transform=axes[0].transAxes)
axes[0].set(xlabel="Recall of accurate readings", ylabel="Precision of accepted readings",
            title="Test precision–recall", xlim=(0, 1), ylim=(0, 1))
axes[0].legend(fontsize=8)
for name, score in {**{n: test_scores[n] for n in models},
                    "constant_training_prevalence": np.full(len(test), constant_probability)}.items():
    probability_rows.append(dict(model=name, n=len(test), good=int(test.agree.sum()),
                                 brier=brier_score_loss(test.agree, score),
                                 log_loss=log_loss(test.agree, score, labels=[0, 1])))
    frame = pd.DataFrame({"score": score, "label": test.agree.to_numpy()})
    boundaries = np.unique(np.r_[0., np.quantile(score, np.linspace(0, 1, 11)), 1.])
    frame["bin"] = pd.cut(frame.score, boundaries, include_lowest=True, duplicates="drop")
    bins = frame.groupby("bin", observed=True).agg(mean_score=("score", "mean"),
                 observed_agreement=("label", "mean"), n=("label", "size"), good=("label", "sum"))
    for interval, row in bins.iterrows():
        calibration_rows.append(dict(model=name, bin=str(interval), **row.to_dict()))
    axes[1].plot(bins.mean_score, bins.observed_agreement, marker="o", label=name, color=colors.get(name, "forestgreen"))
    if name in models:
        axes[2].hist(score, bins=np.linspace(0, 1, 21), histtype="step", label=name, color=colors[name])
axes[1].plot([0, 1], [0, 1], ls="--", color="black", lw=.8)
axes[1].set(xlabel="Mean predicted agreement", ylabel="Observed agreement fraction",
            title="Descriptive calibration; see bin counts", xlim=(0, 1), ylim=(0, 1))
axes[1].legend(fontsize=7)
axes[2].set(xlabel="Predicted agreement score", ylabel="Window count", title="Score support")
axes[2].legend(fontsize=8)
save_figure(fig, "03_pr_and_calibration")
pd.DataFrame(calibration_rows).to_csv(OUTPUT / "calibration_bins.csv", index=False)
probability_report = pd.DataFrame(probability_rows)
probability_report.to_csv(OUTPUT / "probability_benchmarks.csv", index=False)
display(probability_report)

accepted = test_scores[selected] >= thresholds[selected]
fig, axes = plt.subplots(1, 3, figsize=(14, 4), constrained_layout=True)
max_hr = max(200., float(max(test.reference_bpm.max(), test.optical_hr.max())) * 1.03)
for ax, keep, title in [(axes[0], np.ones(len(test), dtype=bool), "Before gate: evaluable"),
                         (axes[1], accepted, f"After gate: {selected}")]:
    if np.any(keep):
        ax.hexbin(test.reference_bpm.to_numpy()[keep], test.optical_hr.to_numpy()[keep],
                  gridsize=45, bins="log", mincnt=1, cmap="viridis", extent=(0, max_hr, 0, max_hr))
    else:
        ax.text(.5, .5, "No accepted readings", ha="center", transform=ax.transAxes)
    ax.plot([0, max_hr], [0, max_hr], color="black", lw=.7)
    for delta in [-TOLERANCE_BPM, TOLERANCE_BPM]:
        ax.plot([0, max_hr], [delta, max_hr + delta], ls="--", color="gray", lw=.6)
    ax.set(xlabel="ECG reference (bpm)", ylabel="Optical estimate (bpm)",
           title=f"{title}\nn={int(np.sum(keep)):,}", xlim=(0, max_hr), ylim=(0, max_hr))
for label, values in [("All evaluable", test.abs_error.to_numpy()),
                       ("Accepted", test.abs_error.to_numpy()[accepted])]:
    if len(values):
        ordered = np.sort(values)
        axes[2].plot(ordered, np.arange(1, len(ordered) + 1) / len(ordered), label=label)
axes[2].axvline(TOLERANCE_BPM, color="black", ls="--", lw=.8)
axes[2].set(xlabel="Absolute HR error (bpm)", ylabel="Fraction at or below error",
            title="Error distribution", ylim=(0, 1))
axes[2].legend(fontsize=8)
save_figure(fig, "04_hr_agreement_and_error")

people_report = pd.DataFrame(by_person).set_index("participant")
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5), constrained_layout=True)
people_report[["false_accept_rate", "good_retained", "error_among_accepted"]].plot.bar(ax=axes[0])
axes[0].set(title="Locked selected gate: test participants", ylabel="Fraction", ylim=(0, 1))
axes[0].tick_params(axis="x", rotation=0)
axes[0].legend(fontsize=8)
hr_report.set_index("reference_hr_band")[["attempts", "evaluable", "accepted_all"]].plot.bar(ax=axes[1])
axes[1].set(title="HR-band support includes ungradable reference", ylabel="Attempted windows")
axes[1].tick_params(axis="x", rotation=30)
axes[1].legend(fontsize=8)
save_figure(fig, "05_participants_and_hr_support")

# Keep every held-out attempt; a missing reference never becomes a 'bad HR' label.
prediction_exports = {}
for partition in ["validation", "test"]:
    exported = windows[windows.split.eq(partition)][["participant", "end_s", "split", "reference_bpm",
        "reference_status", "optical_hr", "optical_status", "features_available", "eligible", "agree", "abs_error"]].copy()
    runtime = windows.loc[exported.index[exported.features_available]]
    runtime_scores = scores_for(runtime) if len(runtime) else {name: np.array([]) for name in test_scores}
    for name in test_scores:
        exported[f"score_{name}"] = np.nan
        exported[f"accepted_{name}"] = False
        exported.loc[runtime.index, f"score_{name}"] = runtime_scores[name]
        exported.loc[runtime.index, f"accepted_{name}"] = runtime_scores[name] >= thresholds[name]
    exported.to_csv(OUTPUT / f"{partition}_window_predictions.csv.gz", index=False, compression="gzip")
    prediction_exports[partition] = exported

for pid in sorted(test_people):
    person = prediction_exports["test"].loc[lambda f: f.participant.eq(pid)].sort_values("end_s")
    start = float(person.end_s.min())
    replay = person[person.end_s.le(start + 600)].copy()
    # Reindex onto the known observation cadence so skipped blocks remain visible.
    expected_time = np.arange(replay.end_s.min(), replay.end_s.max() + HOP_SECONDS / 2, HOP_SECONDS)
    replay = replay.set_index("end_s").reindex(expected_time)
    minutes = (expected_time - start) / 60
    fig, axes = plt.subplots(3, 1, figsize=(11, 6.5), sharex=True, constrained_layout=True)
    axes[0].plot(minutes, replay.reference_bpm, color="black", lw=1, label="ECG reference when qualified")
    axes[0].plot(minutes, replay.optical_hr, color="gray", alpha=.7, lw=.8, label="Optical estimate")
    keep = replay[f"accepted_{selected}"].fillna(False).to_numpy(dtype=bool)
    axes[0].scatter(minutes[keep], replay.optical_hr.to_numpy()[keep], s=8, color="teal", label="Gate accepted")
    axes[0].set(ylabel="HR (bpm)", title=f"{pid}: first ten minutes, selected without outcome filtering")
    axes[0].legend(fontsize=8, loc="upper right")
    axes[1].plot(minutes, replay[f"score_{selected}"], color="teal")
    axes[1].axhline(thresholds[selected], color="darkorange", ls="--", label="Locked acceptance threshold")
    axes[1].set(ylabel="Agreement score", ylim=(-.03, 1.06))
    axes[1].legend(fontsize=8)
    axes[2].step(minutes, replay.features_available.astype(float), where="post", label="Runtime features available")
    axes[2].step(minutes, replay.eligible.astype(float) + 1.3, where="post", label="Runtime and ECG evaluable (+1.3)")
    axes[2].step(minutes, replay[f"accepted_{selected}"].astype(float) + 2.6, where="post", label="Gate accepted (+2.6)")
    axes[2].set(xlabel="Minutes from first attempted window", ylabel="Availability / state")
    axes[2].legend(fontsize=8, loc="upper right")
    save_figure(fig, f"06_replay_{pid}")

selected_result = comparison.loc[comparison.model.eq(selected)].iloc[0]
print(f"Selected gate: {selected}; {selected_result.false_accepts:.0f}/{selected_result.accepted:.0f} "
      "accepted evaluable readings exceeded the tolerance.")
print("Use the CSV denominators and ECG coverage before interpreting any score. "
      "This run does not establish deployable HR reliability or SpO2 reliability.")
print(f"Saved {len(list(FIGURES.glob('*.png')))} diagnostic figures in {FIGURES}")


### License for the heart-rate replay

The Python function `replay_ir_hr` adapts the heart-rate branch of the Maxim MAXREFDES117 reference algorithm. The following notice applies to that implementation. [SparkFun library v1.1.2, original source](https://github.com/sparkfun/SparkFun_MAX3010x_Sensor_Library/blob/v1.1.2/src/spo2_algorithm.cpp)

```text
Copyright (C) 2016 Maxim Integrated Products, Inc., All Rights Reserved.

Permission is hereby granted, free of charge, to any person obtaining a
copy of this software and associated documentation files (the "Software"),
to deal in the Software without restriction, including without limitation
the rights to use, copy, modify, merge, publish, distribute, sublicense,
and/or sell copies of the Software, and to permit persons to whom the
Software is furnished to do so, subject to the following conditions:

The above copyright notice and this permission notice shall be included
in all copies or substantial portions of the Software.

THE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND, EXPRESS
OR IMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF
MERCHANTABILITY, FITNESS FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT.
IN NO EVENT SHALL MAXIM INTEGRATED BE LIABLE FOR ANY CLAIM, DAMAGES
OR OTHER LIABILITY, WHETHER IN AN ACTION OF CONTRACT, TORT OR OTHERWISE,
ARISING FROM, OUT OF OR IN CONNECTION WITH THE SOFTWARE OR THE USE OR
OTHER DEALINGS IN THE SOFTWARE.

Except as contained in this notice, the name of Maxim Integrated
Products, Inc. shall not be used except as stated in the Maxim Integrated
Products, Inc. Branding Policy.

The mere transfer of this software does not imply any licenses
of trade secrets, proprietary technology, copyrights, patents,
trademarks, maskwork rights, or any other form of intellectual
property whatsoever. Maxim Integrated Products, Inc. retains all
ownership rights.
```